In [0]:
catalog = "neobank"
volume_schema = "neobank"
volume = "neobank"
base = f"/Volumes/{catalog}/{volume_schema}/{volume}/raw"

tables = ["accounts", "branches", "credit_bureau_reports", "customers", "payment_gateway_logs", "transactions"]

for t in tables:
    print(t, [f.name for f in dbutils.fs.ls(f"{base}/{t}")])

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.bronze")

In [0]:
df = spark.read.option("header", "true").option("inferSchema", "true").csv(f"{base}/customers")
df.printSchema()
display(df.limit(5))

In [0]:
from pyspark.sql import functions as F

for t in tables:
    df = (
        spark.read.option("header", "true")
        .option("inferSchema", "true")
        .csv(f"{base}/{t}")
        .select("*", F.current_timestamp().alias("insert_timestamp"))
    )

    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.bronze.{t}")

    print(f"{t}: {df.count()} rows")

In [0]:
for t in tables:
    print("=" * 40, t)
    spark.table(f"{catalog}.bronze.{t}").printSchema()